# Data Drift and Successive Retraining

A model in production degrades over time: the data changes (data drift) and so does the relation between inputs and output (concept drift).

In this demonstration, we forecast the number of bikes rented every hour in Washington (2011–2012) and compare three retraining policies:

- **Never**: the initial model stays in production
- **Every month**: retrain on all available data, every month
- **On degradation**: retrain only when the measured error clearly increases

In [ ]:
!pip install -q "evidently>=0.7,<0.8"

import warnings

import matplotlib.pyplot as plt
import pandas as pd
from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")

## The Data

The [*Bike Sharing*](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) dataset gives, for every hour, the weather, the calendar and the number of rentals (`cnt`, the value to predict).

In [ ]:
df = pd.read_csv(
    "https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-sharing/hour.csv", parse_dates=["dteday"]
)
df["month"] = df["dteday"].dt.to_period("M")

FEATURES = ["temp", "atemp", "hum", "windspeed", "hr", "weekday", "workingday",
            "holiday", "weathersit"]
WEATHER = ["temp", "atemp", "hum", "windspeed"]
TARGET = "cnt"
df[["dteday", *FEATURES, TARGET]].head()

## A Concept Drift: the Service Becomes Popular

With the same weather, many more bikes are rented in 2012 than in 2011: the relation between inputs and output has changed. No input variable lets the model see it.

In [ ]:
YLABEL_RENTALS = "Rentals per hour (mean)"
XLABEL_MONTH = "Month"
TITLE_CONCEPT = "Mean rentals per month"
MSG_TRAINED = "Model trained on {} hours of data"
LEGEND = ["Never", "Every month", "On degradation"]
YLABEL_MAE = "MAE (bikes per hour)"
TITLE_POLICIES = "Monthly error by retraining policy (dotted: retrainings)"
COL_MAE = "Mean MAE"
COL_RETRAINS = "Retrainings"

monthly = df.groupby(["yr", "mnth"])[TARGET].mean().unstack(0)
monthly.columns = ["2011", "2012"]
monthly.plot(kind="bar", figsize=(10, 4), ylabel=YLABEL_RENTALS, xlabel=XLABEL_MONTH)
plt.title(TITLE_CONCEPT)
plt.show()

## A First Model, Trained on January and February 2011

In [ ]:
def train(data: pd.DataFrame) -> RandomForestRegressor:
    model = RandomForestRegressor(n_estimators=100, min_samples_leaf=5, n_jobs=-1,
                                  random_state=0)
    model.fit(data[FEATURES], data[TARGET])
    return model


months = sorted(df["month"].unique())
initial = df[df["month"].isin(months[:2])]
model = train(initial)
print(MSG_TRAINED.format(len(initial)))

## Detecting a Data Drift with `evidently`

Let's compare the weather of July 2011 to that of the training data:

In [ ]:
definition = DataDefinition(numerical_columns=WEATHER)
july = df[df["month"] == pd.Period("2011-07")]
report = Report([DataDriftPreset()])
run = report.run(
    reference_data=Dataset.from_pandas(initial[WEATHER], data_definition=definition),
    current_data=Dataset.from_pandas(july[WEATHER], data_definition=definition),
)
run

The drift is massive: July is much warmer than winter. Yet it is an **expected** drift (the seasons): a drift alert is a signal to analyze, not a sufficient reason to retrain. The deciding criterion is the measured performance, as soon as the true values are known.

## Comparing Three Retraining Policies

Every month, the production model predicts, then the true rentals are known: we measure the mean absolute error (MAE, in bikes per hour) and each policy decides whether to retrain.

In [ ]:
def mae(model: RandomForestRegressor, data: pd.DataFrame) -> float:
    return mean_absolute_error(data[TARGET], model.predict(data[FEATURES]))


never = monthly_model = perf_model = model
seen = initial.copy()
baseline = None
rows = []
for month in months[2:]:
    batch = df[df["month"] == month]
    # 1. The month's predictions are made, then the true values arrive
    row = {"month": str(month), "never": mae(never, batch),
           "monthly": mae(monthly_model, batch), "on_degradation": mae(perf_model, batch)}
    seen = pd.concat([seen, batch])
    # 2. "Every month" policy: always retrain
    monthly_model = train(seen)
    # 3. "On degradation" policy: retrain when the error exceeds by 25%
    #    the one measured right after the last training
    if baseline is None:
        baseline = row["on_degradation"]
    row["retrained"] = row["on_degradation"] > 1.25 * baseline
    if row["retrained"]:
        perf_model = train(seen)
        baseline = None
    rows.append(row)

results = pd.DataFrame(rows).set_index("month")
results.round(1)

In [ ]:
ax = results[["never", "monthly", "on_degradation"]].plot(figsize=(11, 4), marker="o")
ax.legend(LEGEND)
for i, retrained in enumerate(results["retrained"]):
    if retrained:
        ax.axvline(i, color="grey", linestyle=":")
ax.set_ylabel(YLABEL_MAE)
plt.title(TITLE_POLICIES)
plt.show()

In [ ]:
summary = pd.DataFrame({
    COL_MAE: results[["never", "monthly", "on_degradation"]].mean().round(1).values,
    COL_RETRAINS: [0, len(results), int(results["retrained"].sum())],
}, index=LEGEND)
summary

## Takeaways

- Without retraining, the error explodes: from 25 to more than 200 bikes per hour
- Retraining every month gives the best error, but costs 22 trainings, deployments and validations
- Retraining on degradation gets most of the gain with a handful of retrainings
- In every case, you must **measure** performance in production: it is what triggers action